# Install conda on your Colab environment

Ignore this first cell if you are running the notebook in a local environment.

One can still run it locally but it will have no effect.

In [ ]:
# Run this cell first - it will install a conda distribution (mamba)
# on your Drive then restart the kernel automatically 
# (don't worry about the crashing/restarting kernel messages)
# It HAS to be runned FIRST everytime you use the notebook in colab

# import os
# import sys
# RunningInCOLAB  = 'google.colab' in str(get_ipython())

# if RunningInCOLAB:
#     !pip install -q condacolab
#     import condacolab
#     condacolab.install()

# Set up your Colab or local environment
# Then import libraries

Run this cell in both cases of use (local or Colab)

In [1]:
import os
import sys
RunningInCOLAB  = 'google.colab' in str(get_ipython())

if RunningInCOLAB:
    
    # Check everything is fine with conda in Colab
    import condacolab
    condacolab.check()
    
    # Mount your drive environment in the colab runtime
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    
    # Change this variable to your path on Google Drive to which the repo has been cloned
    # If you followed the colab notebook 'repo_cloning.ipynb', nothing to change here
    repo_path_in_drive = '/content/drive/My Drive/Github/amn_release/'
    # Change directory to your repo cloned in your drive
    DIRECTORY = repo_path_in_drive
    os.chdir(repo_path_in_drive)
    # Copy the environment given in the environment_amn_light.yml
    !mamba env update -n base -f environment_amn_light.yml
    
    # This is one of the few Colab-compatible font
    font = 'Liberation Sans'
    
else:
    
    # In this case the local root of the repo is our working directory
    DIRECTORY = './'
    font = 'arial'

# printing the working directory files. One can check you see the same folders and files as in the git webpage.
print(os.listdir(DIRECTORY))

from Library.Build_Model import *

# We declare this function here and not in the
# function-storing python file to modify it easily
# as it can change the printouts of the methods
def printout(filename, Stats, model, time): 
    # printing Stats
    print('Stats for %s CPU-time %.4f' % (filename, time))
    print('R2 = %.4f (+/- %.4f) Constraint = %.4f (+/- %.4f)' % \
          (Stats.train_objective[0], Stats.train_objective[1],
           Stats.train_loss[0], Stats.train_loss[1]))
    print('Q2 = %.4f (+/- %.4f) Constraint = %.4f (+/- %.4f)' % \
          (Stats.test_objective[0], Stats.test_objective[1],
           Stats.test_loss[0], Stats.test_loss[1]))

['.git', '.gitignore', '.venv', 'Build_Dataset.ipynb', 'Build_Dataset_KO.ipynb', 'Build_Experimental.ipynb', 'Build_Model_AMN.ipynb', 'Build_Model_AMN_KO.ipynb', 'Build_Model_ANN_Dense.ipynb', 'Build_Model_MM.ipynb', 'Build_Model_RC.ipynb', 'Dataset_experimental', 'Dataset_input', 'Dataset_model', 'Duplicate_Model.ipynb', 'Figures', 'Figures.ipynb', 'LICENSE', 'Library', 'README.md', 'Reservoir', 'Result', 'Tutorial.ipynb', 'environment_amn.yml', 'environment_amn_light.yml', 'pyproject.toml', 'uv.lock']


2026-01-15 14:59:33.861604: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-01-15 14:59:33.861639: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-01-15 14:59:58.791060: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-01-15 14:59:58.791120: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (l4.nibi.sharcnet): /proc/driver/nvidia/version does not exist


# Hybrid models

# Examples of trainable AMN models (neural and mechanistic)

# Using FBA simulated training sets or experimental datasets

Each cell is showing a different setup: 

- using either E. coli core simulations, iML1515 simulations, or the experimental dataset as the training set;
- using either LP, QP or Wt as the mechanistic layer of the AMNs

### AMN with QP solver on *E. coli* core FBA simulated training set

In [2]:
# Create, train and evaluate AMN_QP o models with FBA simulated training set for E. coli core
# with EB or UB with a mechanistic layer

# What you can change 
seed = 10
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'e_coli_core_UB_100' # can change EB by UB
timestep = 4
# End of What you can change
  
# Create model 90% for training 10% for testing
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
                     objective=['BIOMASS_Ecoli_core_w_GAM'],  
                     model_type='AMN_QP', 
                     timestep = timestep, learn_rate=0.01,
                     scaler=True,
                     n_hidden = 1, hidden_dim = 50,
                     epochs=50, xfold=5,
                     verbose=True)
ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y = Xtrain, Ytrain
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=False)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
reservoir.save(reservoirfile)
reservoir.load(reservoirfile)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    X, Y = model_input(reservoir, verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)
    

number of reactions:  154 154
number of metabolites:  72
filtered measurements size:  1
training file: ./Dataset_model/e_coli_core_UB_100
model type: AMN_QP
model scaler: 1.0
model input dim: 20
model output dim: 1
model medium bound: UB
timestep: 4
training set size (100, 20) (100, 1)
nbr hidden layer: 1
hidden layer size: 50
activation function: relu
gradient learn rate: 0.01
gradient decay rate: 0.9
training epochs: 50
training regression: True
training learn rate: 0.001
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False


2026-01-15 15:00:15.421176: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


nbr parameters: 8904


2026-01-15 15:00:19.158164: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


---------- 1
---------- 1
train = 0.82 test = 0.82 loss-train = 0.005319 loss-test = 0.005085 iter=0
nbr parameters: 8904
---------- 1
---------- 1
train = 0.78 test = 0.81 loss-train = 0.005316 loss-test = 0.005796 iter=0
nbr parameters: 8904
---------- 1
---------- 1
train = 0.77 test = 0.70 loss-train = 0.003773 loss-test = 0.003476 iter=0
nbr parameters: 8904
---------- 1
---------- 1
train = 0.79 test = 0.48 loss-train = 0.003490 loss-test = 0.003560 iter=0
nbr parameters: 8904
---------- 1
---------- 1
train = 0.76 test = 0.62 loss-train = 0.004921 loss-test = 0.004873 iter=0
---------- 1
Stats for e_coli_core_UB_100_AMN_QP CPU-time 40.1926
R2 = 0.7854 (+/- 0.0216) Constraint = 0.0046 (+/- 0.0008)
Q2 = 0.6855 (+/- 0.1266) Constraint = 0.0046 (+/- 0.0009)
./Reservoir/e_coli_core_UB_100_AMN_QP_param.csv
training file: ./Dataset_model/e_coli_core_UB_100
model type: AMN_QP
model scaler: 10.0
model input dim: 20
model output dim: 312
model medium bound: UB
timestep: 4
training set siz

### AMN with LP solver on *E. coli* core FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_LP models with FBA simulated training set for E. coli core
# with UB with a mechanistic layer

# What you can change 
seed = 10
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'e_coli_core_UB_100' # can change EB by UB
timestep = 4
# End of What you can change
  
# Create model 90% for training 10% for testing
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
                     objective=['BIOMASS_Ecoli_core_w_GAM'],  
                     model_type='AMN_LP', 
                     timestep = timestep, learn_rate=1.0e-6,
                     scaler=True,
                     n_hidden = 1, hidden_dim = 50,
                     epochs= 50, xfold=5,
                     verbose=True)
ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), 
                      replace=False)
# LP keeps track of boundary fluxes in b_int and b_ext
# and these are different in EB or UB modes
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
btest = model.b_ext[ID,:] if model.mediumbound == 'UB' else model.b_int[ID,:]
bint  = model.b_int if model.mediumbound == 'UB' else model.b_ext
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
btrain = np.delete(model.b_ext, ID, axis=0) if model.mediumbound == 'UB' \
else np.delete(model.b_int, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y, model.b_ext, model.b_int = Xtrain, Ytrain, btrain, bint
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=False)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
reservoir.save(reservoirfile)
reservoir.load(reservoirfile) 
# Issue for loading AMN-LP models: takes about 15 minutes (when QP takes a few)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    reservoir.b_ext, reservoir.b_int =  btest, bint
    X, Y = model_input(reservoir,verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)
    

number of reactions:  154 154
number of metabolites:  72
filtered measurements size:  1
training file: ./Dataset_model/e_coli_core_UB_100
model type: AMN_LP
model scaler: 1.0
model input dim: 20
model output dim: 1
model medium bound: UB
timestep: 4
training set size (100, 20) (100, 1)
nbr hidden layer: 1
hidden layer size: 50
activation function: relu
training epochs: 50
training regression: True
training learn rate: 0.001
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False
nbr parameters: 25152
---------- 1
---------- 1
train = 0.75 test = 0.78 loss-train = 0.004792 loss-test = 0.004545 iter=0
nbr parameters: 25152
---------- 1
---------- 1
train = 0.74 test = 0.75 loss-train = 0.003338 loss-test = 0.003617 iter=0
nbr parameters: 25152
---------- 1
---------- 1
train = 0.70 test = 0.69 loss-train = 0.004940 loss-test = 0.004615 iter=0
nbr parameters: 25152
---------- 1
---------- 1
train = 0.72 test = 0.33 loss-tr

: 

### AMN with QP solver on iML1515 FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_QP models with FBA simulated training set for iML1515
# with EB or UB with a mechanistic layer 
# This cell takes several hours to execute

# What you can change 
seed = 1
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'iML1515_UB' # can change EB by UB
timestep = 4
# End of What you can change
  
# Create model 
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_QP',
              scaler = True,
              timestep = timestep, learn_rate=0.01,
              n_hidden = 1, hidden_dim = 500,
              epochs = 25, xfold = 5, 
              verbose=True) 
ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y = Xtrain, Ytrain
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=2)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
reservoir.save(reservoirfile)
reservoir.load(reservoirfile)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    X, Y = model_input(reservoir, verbose=False)
    reservoir.model.b_ext = btest
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)
 

number of reactions:  550 550
number of metabolites:  1083
filtered measurements size:  1
training file: ./Dataset_model/iML1515_UB
model type: AMN_QP
model scaler: 1.0
model input dim: 38
model output dim: 1
model medium bound: UB
timestep: 4
training set size (11000, 38) (11000, 1)
nbr hidden layer: 1
hidden layer size: 500
activation function: relu
gradient learn rate: 0.01
gradient decay rate: 0.9
training epochs: 25
training regression: True
training learn rate: 0.001
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False
AMN scaler 11.0
QP input shape (9900, 38) (9900, 4)
-------train [   0    1    3 ... 9897 9898 9899] (7920, 38) (7920, 4)
-------test  [   2    4    6 ... 9883 9891 9893] (1980, 38) (1980, 4)
----------------------------------- AMN_QP
Dense layer n_hidden, hidden_dim, output_dim, activation, trainable: 1 500 550 relu True
AMN output shapes for PoutV, SV, PinV, Vpos, V, outputs (None, 1) (None, 1)

### AMN with QP solver on iJN1463 FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_QP models with FBA simulated training set for P. putida iJN1463
# with UB with a mechanistic layer 
# This cell takes several hours to execute

# What you can change 
seed = 1
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'IJN1463_10_UB' # can change EB by UB
timestep = 4
# End of What you can change
  
# Create model 
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_KT2440_WT3'], 
              model_type = 'AMN_QP',
              scaler = True,
              timestep = timestep, learn_rate=0.01,
              n_hidden = 1, hidden_dim = 500, batch_size=100,
              epochs = 500, xfold = 5, 
              verbose=True) 
ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y = Xtrain, Ytrain
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=False)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
reservoir.save(reservoirfile)
reservoir.load(reservoirfile)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    X, Y = model_input(reservoir, verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)

number of reactions:  2135 2135
number of metabolites:  1637
filtered measurements size:  1
training file: ./Dataset_model/IJN1463_10_UB
model type: AMN_QP
model scaler: 1.0
model input dim: 196
model output dim: 1
model medium bound: UB
timestep: 4
training set size (4860, 196) (4860, 1)
nbr hidden layer: 1
hidden layer size: 500
activation function: relu
gradient learn rate: 0.01
gradient decay rate: 0.9
training epochs: 500
training regression: True
training learn rate: 0.001
training dropout: 0.25
training batch size: 100
training validation iter: 0
training xfold: 5
training early stopping: False
nbr parameters: 1168135
110/110 [==============================] - 3s 26ms/step
---------- 1
28/28 [==============================] - 1s 26ms/step
---------- 1
train = 0.99 test = 0.98 loss-train = 0.000009 loss-test = 0.000009 iter=0
nbr parameters: 1168135
110/110 [==============================] - 3s 25ms/step
---------- 1
28/28 [==============================] - 1s 28ms/step
---------

### AMN with LP solver on iML1515 FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_LP models with FBA simulated training set for iML1515
# with UB with a mechanistic layer 
# This cell takes several hours to execute

# What you can change 
seed = 1
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'iML1515_UB'
timestep = 4
# End of What you can change
  
# Create model 
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_LP',
              scaler = True,
              timestep = timestep, learn_rate=1.0e-6,
              n_hidden = 1, hidden_dim = 250,
              epochs = 25, xfold = 5, 
              verbose=True) 

ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), 
                      replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
btest = model.b_ext[ID,:]
bint = model.b_int
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
btrain = np.delete(model.b_ext, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y, model.b_ext, model.b_int = Xtrain, Ytrain, btrain, bint
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=2)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
reservoir.save(reservoirfile)
# reservoir.load(reservoirfile)
# Issue for loading AMN-LP models: takes about 15 minutes (when QP takes a few)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    reservoir.b_ext, reservoir.b_int =  btest, bint
    X, Y = model_input(reservoir,verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)


number of reactions:  550 550
number of metabolites:  1083
filtered measurements size:  1
training file: ./Dataset_model/iML1515_UB
model type: AMN_LP
model scaler: 1.0
model input dim: 38
model output dim: 1
model medium bound: UB
timestep: 4
training set size (11000, 38) (11000, 1)
nbr hidden layer: 1
hidden layer size: 250
activation function: relu
training epochs: 25
training regression: True
training learn rate: 0.001
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False
AMN scaler 11.0
LP input shape (9900, 3235) (9900, 4)
-------train [   0    1    3 ... 9897 9898 9899]
-------test  [   2    4    6 ... 9883 9891 9893]
Dense layer n_hidden, hidden_dim, output_dim, activation, trainable: 1 250 550 relu True
Dense layer n_hidden, hidden_dim, output_dim, activation, trainable: 1 250 2716 linear True
AMN output shapes for PoutV, SV, PinV, Vpos, V, outputs (None, 1) (None, 1) (None, 1) (None, 1) (None, 550) (None, 11

### AMN-Wt on *E. coli* core FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_Wt models with FBA simulated training set for E. coli core
# with UB (not working with M1 chips)

# What you can change 
seed = 10
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'e_coli_core_UB' # can change EB by UB
timestep = 4
# End of What you can change
  
# Create model 90% for training 10% for testing
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
                     objective=['BIOMASS_Ecoli_core_w_GAM'],  
                     model_type='AMN_Wt', 
                     timestep = timestep,
                     n_hidden = 1, hidden_dim = 50,
                     scaler=True,
                     train_rate=1e-2,
                     epochs=500, xfold=5,
                     verbose=True)
ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0)
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y = Xtrain, Ytrain
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=False)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
#reservoir.save(reservoirfile)
#reservoir.load(reservoirfile)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    X, Y = model_input(reservoir, verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)

number of reactions:  154 154
number of metabolites:  72
filtered measurements size:  1
training file: ./Dataset_model/e_coli_core_UB
model type: AMN_Wt
model scaler: 1.0
model input dim: 20
model output dim: 1
model medium bound: UB
timestep: 4
training set size (1000, 20) (1000, 1)
nbr hidden layer: 1
hidden layer size: 50
activation function: relu
training epochs: 500
training regression: True
training learn rate: 0.01
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False
nbr parameters: 13262
6/6 [==============================] - 0s 2ms/step
train = 0.99 test = 0.99 loss-train = 0.000919 loss-test = 0.000912 iter=0
nbr parameters: 13262
6/6 [==============================] - 0s 2ms/step
train = 0.99 test = 0.99 loss-train = 0.000933 loss-test = 0.000990 iter=0
nbr parameters: 13262
6/6 [==============================] - 0s 2ms/step
train = 0.99 test = 0.99 loss-train = 0.000908 loss-test = 0.000904 iter=0
nbr par

### AMN-Wt on iML1515 FBA simulated training set

In [ ]:
# Create, train and evaluate AMN_QP models with FBA simulated training set for iML1515
# with EB or UB with a mechanistic layer 
# This cell takes several hours to execute

# What you can change 
seed = 1
ratio_test = 0.1 # part of the training set removed for test
np.random.seed(seed=seed)  
trainname = 'iML1515_UB'
timestep = 4
# End of What you can change
  
# Create model 
trainingfile = DIRECTORY+'Dataset_model/'+trainname
model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_Wt',
              scaler = True,
              timestep = timestep, 
              n_hidden = 1, hidden_dim = 500,
              train_rate=1e-2,
              epochs = 100, xfold = 5, 
              verbose=True) 

ID = np.random.choice(model.X.shape[0], 
                      size=int(model.X.shape[0]*ratio_test), 
                      replace=False)
Xtest,  Ytest  = model.X[ID,:], model.Y[ID,:]
Xtrain, Ytrain = np.delete(model.X, ID, axis=0), np.delete(model.Y, ID, axis=0) 
model.printout()

# Train and evaluate
reservoirname = trainname+'_'+model.model_type
reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
start_time = time.time()
model.X, model.Y = Xtrain, Ytrain
reservoir, pred, stats, _ = train_evaluate_model(model, verbose=2)
delta_time = time.time() - start_time

# Printing cross-validation results
printout(reservoirname, stats, model, delta_time)

# Save, reload and run idependent test set
#reservoir.save(reservoirfile)
#reservoir.load(reservoirfile)
reservoir.printout()
if len(Xtest) > 0:
    start_time = time.time()
    reservoir.X, reservoir.Y = Xtest, Ytest
    X, Y = model_input(reservoir,verbose=False)
    pred, stats = evaluate_model(reservoir.model, X, Y, reservoir, verbose=False)
    delta_time = time.time() - start_time
    printout('Test set', stats, model, delta_time)

number of reactions:  550 550
number of metabolites:  1083
filtered measurements size:  1
training file: ./Dataset_model/iML1515_UB
model type: AMN_Wt
model scaler: 1.0
model input dim: 38
model output dim: 1
model medium bound: UB
timestep: 4
training set size (11000, 38) (11000, 1)
nbr hidden layer: 1
hidden layer size: 500
activation function: relu
training epochs: 100
training regression: True
training learn rate: 0.01
training dropout: 0.25
training batch size: 5
training validation iter: 0
training xfold: 5
training early stopping: False
AMN scaler 11.0
Wt input shape (9900, 4, 38) (9900, 4)
-------train [   0    1    3 ... 9897 9898 9899]
-------test  [   2    4    6 ... 9883 9891 9893]
number of reactions:  550 550
number of metabolites:  1083
filtered measurements size:  1
Model: "model_11"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to             

### AMN-QP with experimental training set

In [ ]:
# Create, train and evaluate AMN_QP models on experimental training set with UB 
# Repeat the process with different seeds
# This cell takes several hours to execute

Maxloop, Q2, PRED = 3, [], []

for Nloop in range(Maxloop):
    # What you can change 
    seed = Nloop+1
    np.random.seed(seed=seed)  
    trainname = 'iML1515_EXP_UB' 
    timestep = 4
    # End of What you can change
  
    # Create model 100% for training 0% for testing
    trainingfile = DIRECTORY+'Dataset_model/'+trainname
    model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_QP',
              scaler = True,
              timestep = timestep, learn_rate=0.001,
              n_hidden = 1, hidden_dim = 500,
              #train_rate = 1.0e-2,
              epochs = 1000, xfold = 10, 
              verbose=True) 

    # Train and evaluate
    reservoirname = trainname +'_'+model.model_type
    reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
    start_time = time.time()
    reservoir, pred, stats, _ = train_evaluate_model(model, verbose=True)
    delta_time = time.time() - start_time

    # Printing cross-validation results
    printout(reservoirname, stats, model, delta_time)
    r2 = r2_score(model.Y, pred[:,0], multioutput='variance_weighted')
    print('Iter', Nloop, 'Collated Q2', r2)
    Q2.append(r2)
    PRED.append(pred[:,0])

# Save in Result folder
Q2, PRED = np.asarray(Q2), np.asarray(PRED)
print('Averaged Q2 = %.4f (+/- %.4f)' % (np.mean(Q2), np.std(Q2)))
filename = DIRECTORY+'Result/'+reservoirname+'_Q2.csv'
np.savetxt(filename, Q2, delimiter=',')
filename = DIRECTORY+'Result/'+reservoirname+'_PRED.csv'
np.savetxt(filename, PRED, delimiter=',')


### AMN-LP with experimental training set

In [ ]:
# Create, train and evaluate AMN_QP models on experimental training set with UB 
# Repeat the process with different seeds
# This cell takes several hours to execute

Maxloop, Q2, PRED = 3, [], []

for Nloop in range(Maxloop):
    # What you can change 
    seed = Nloop+1
    np.random.seed(seed=seed)  
    trainname = 'iML1515_EXP_UB' 
    timestep = 4
    # End of What you can change
  
    # Create model 100% for training 0% for testing
    trainingfile = DIRECTORY+'Dataset_model/'+trainname
    model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_LP',
              scaler = True,
              timestep = timestep, learn_rate=0.001,
              n_hidden = 1, hidden_dim = 500,
              #train_rate = 1.0e-2,
              epochs = 1000, xfold = 10, 
              verbose=True) 

    # Train and evaluate
    reservoirname = trainname +'_'+model.model_type
    reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
    start_time = time.time()
    reservoir, pred, stats, _ = train_evaluate_model(model, verbose=2)
    delta_time = time.time() - start_time

    # Printing cross-validation results
    printout(reservoirname, stats, model, delta_time)
    r2 = r2_score(model.Y, pred[:,0], multioutput='variance_weighted')
    print('Iter', Nloop, 'Collated Q2', r2)
    Q2.append(r2)
    PRED.append(pred[:,0])

# Save in Result folder
Q2, PRED = np.asarray(Q2), np.asarray(PRED)
print('Averaged Q2 = %.4f (+/- %.4f)' % (np.mean(Q2), np.std(Q2)))
filename = DIRECTORY+'Result/'+reservoirname+'_Q2.csv'
np.savetxt(filename, Q2, delimiter=',')
filename = DIRECTORY+'Result/'+reservoirname+'_PRED.csv'
np.savetxt(filename, PRED, delimiter=',')


### AMN-Wt with experimental training set

In [ ]:
# Create, train and evaluate AMN_Wt models on experimental training set with UB 
# Repeat the process with different seeds
# This cell takes several hours to execute

Maxloop, Q2, PRED = 3, [], []

for Nloop in range(Maxloop):
    # What you can change 
    seed = Nloop+1
    np.random.seed(seed=seed)  
    trainname = 'iML1515_EXP_UB' 
    timestep = 4
    # End of What you can change
  
    # Create model 100% for training 0% for testing
    trainingfile = DIRECTORY+'Dataset_model/'+trainname
    model = Neural_Model(trainingfile = trainingfile, 
              objective=['BIOMASS_Ec_iML1515_core_75p37M'], 
              model_type = 'AMN_Wt',
              scaler = True,
              timestep = timestep,
              n_hidden = 1, hidden_dim = 500,
              #train_rate = 1.0e-2,
              epochs = 1000, xfold = 10, 
              verbose=True) 

    # Train and evaluate
    reservoirname = trainname +'_'+model.model_type
    reservoirfile = DIRECTORY+'Reservoir/'+reservoirname
    start_time = time.time()
    reservoir, pred, stats, _ = train_evaluate_model(model, verbose=True)
    delta_time = time.time() - start_time

    # Printing cross-validation results
    printout(reservoirname, stats, model, delta_time)
    r2 = r2_score(model.Y, pred[:,0], multioutput='variance_weighted')
    print('Iter', Nloop, 'Collated Q2', r2)
    Q2.append(r2)
    PRED.append(pred[:,0])

# Save in Result folder
Q2, PRED = np.asarray(Q2), np.asarray(PRED)
print('Averaged Q2 = %.4f (+/- %.4f)' % (np.mean(Q2), np.std(Q2)))
filename = DIRECTORY+'Result/'+reservoirname+'_Q2.csv'
np.savetxt(filename, Q2, delimiter=',')
filename = DIRECTORY+'Result/'+reservoirname+'_PRED.csv'
np.savetxt(filename, PRED, delimiter=',')
